In [1]:
import mysql.connector
from mysql.connector import Error
import pandas as pd

In [9]:
def create_db_connection(host_name, user_name, user_password, db_name):
    connection = None
    try:
        connection = mysql.connector.connect(
            host=host_name,
            user=user_name,
            passwd=user_password,
            database = db_name
        )
        print("MySQL Database connection successful")
    except Error as err:
        print(f"Error: '{err}'")

    return connection


def execute_query(connection, query, values=None):
    cursor = connection.cursor()
    try:
        if values:
            if isinstance(values[0], (list, tuple)):  
                    cursor.executemany(query, values)
            else: 
                cursor.execute(query, values)
        else:
            cursor.execute(query)
        connection.commit()
        print("Query successful")
    except Error as err:
        print(f"Error: '{err}'")


In [ ]:
file_path = "./crawl_sendo_api/sendo_products.csv"  
df = pd.read_csv(file_path)
df_seller = df[['sellerID', 'sellerName', 'sellerURL', 'sellerRating', 'sellerType','sellerLocation']]
df_product = df[['productID', 'productName', 'productURL', 'productImage',
       'productRating', 'productNumRate', 'productNumSold', 'productBrand',
       'categoryID', 'productOrgPrice', 'productCrtPrice', 'sellerID']]
df_category = df[['categoryID', 'categoryInfo']]


In [ ]:
df_seller.drop_duplicates(subset=['sellerID'], inplace=True)

In [ ]:
df_seller.head(100)

In [ ]:
connection =create_db_connection("localhost", "root", "031103", "ecommerce_db")

# import data into Seller
for index, row in df_seller.iterrows():
    insert_query = """
    INSERT INTO Seller (SellerID, Name, Link, Rating, Type, Location)
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    values = (
        row["sellerID"], row["sellerName"], row["sellerURL"], 
        row["sellerRating"], row["sellerType"], row["sellerLocation"]
    )
    
    execute_query(connection, insert_query, values)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

In [ ]:
connection =create_db_connection("localhost", "root", "031103", "ecommerce_db")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")

# import data into Product
for index, row in df_product.iterrows():
    insert_query = """
    INSERT INTO Product (ProductID, Name, URLProduct, URLImage, Rating, Num_rate, Num_sold, Brand, Category, OriginalPrice, PlatformID, Seller)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    
    values = (
        row["productID"], row["productName"], row["productURL"], row["productImage"], 
        row["productRating"], row["productNumRate"], row["productNumSold"], 
        row["productBrand"], row["categoryID"], row["productOrgPrice"], 
        'SD', row["sellerID"]  # PlatformID = SD -> Sendo
    )
    
    execute_query(connection, insert_query, values)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

In [ ]:
# insert data into Category
connection =create_db_connection("localhost", "root", "031103", "ecommerce_db")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")


# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")